# 03 — Classical ML End-to-End (Phase 3, Milestone 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udaysharmadev/Ai-Roadmap/blob/main/notebooks/03_sklearn_end_to_end.ipynb)

**Maps to:** `docs/machine-learning-roadmap.md > Phase 3` + `README Milestone 3`  
**Dataset:** `data/samples/titanic_sample.csv` (61 rows, offline)  
**Goal:** train a real classifier with a leak-free pipeline and evaluate it with our from-scratch metrics.

Flow: clean → split → LogReg pipeline → RandomForest comparison → our metrics vs sklearn → confusion matrix → save model.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix as sk_cm
from sklearn.pipeline import Pipeline

from ai_roadmap.data_wrangling import impute_missing
from ai_roadmap.features import (
    infer_feature_types,
    make_classification_pipeline,
    train_test_split_df,
)
from ai_roadmap.metrics import classification_report, f1

print("imports ok")

## 1. Load + clean (reuse Chunk 2)

We impute but keep the target column — `clean_dataframe` would also work, this shows the manual path.

In [ ]:
df = pd.read_csv(ROOT / "data/samples/titanic_sample.csv")
print(f"raw: {df.shape}, missing: {int(df.isna().sum().sum())}, dupes: {int(df.duplicated().sum())}")

df = impute_missing(df).drop_duplicates().reset_index(drop=True)
print(f"clean: {df.shape}, missing: {int(df.isna().sum().sum())}")

num, cat = infer_feature_types(df, target="survived", exclude=["passenger_id"])
print(f"numeric: {num}")
print(f"categorical: {cat}")

## 2. Stratified split (reproducible)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split_df(df, "survived", test_size=0.25, random_state=42)
print(f"train: {X_train.shape}  test: {X_test.shape}")
print(f"train positive rate: {y_train.mean():.2f} | test positive rate: {y_test.mean():.2f}")

## 3. Baseline: Logistic Regression pipeline (no leakage)

Imputation + scaling + one-hot all happen *inside* the pipeline, fitted on train only.

In [ ]:
logreg = make_classification_pipeline(num, cat, random_state=42)
logreg.fit(X_train, y_train)
pred = logreg.predict(X_test)
print(f"test accuracy: {logreg.score(X_test, y_test):.3f}")
print(classification_report(y_test.to_numpy(), pred))

## 4. Challenger: RandomForest on the same features

Same preprocessor, different estimator — the correct way to compare models.

In [ ]:
from ai_roadmap.features import build_preprocessor

rf = Pipeline(
    steps=[
        ("preprocess", build_preprocessor(num, cat)),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ]
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print(f"logreg F1: {f1(y_test.to_numpy(), pred):.3f}")
print(f"forest F1: {f1(y_test.to_numpy(), pred_rf):.3f}")

## 5. Confusion matrix (ours matches sklearn)

In [ ]:
import numpy as np

cm = sk_cm(y_test, pred)
print(cm)  # [[TN FP] [FN TP]]

fig, ax = plt.subplots(figsize=(4, 3.5))
ax.matshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center")
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("Titanic — confusion matrix (logreg)")
fig.tight_layout()
out = ROOT / "outputs"
out.mkdir(exist_ok=True)
fig.savefig(out / "titanic_confusion.png", dpi=120)
plt.close(fig)
print(f"saved: {out / 'titanic_confusion.png'}")

## 6. Save the model (Chunk 6 will serve this)

`joblib` preserves the whole pipeline — preprocessor + classifier — so deployment needs one file.

In [ ]:
import joblib

model_path = out / "titanic_logreg.joblib"
joblib.dump(logreg, model_path)
print(f"saved: {model_path} ({model_path.stat().st_size / 1024:.1f} KB)")

# Reload sanity check: same predictions after a round-trip
reloaded = joblib.load(model_path)
assert (reloaded.predict(X_test) == pred).all()
print("reload check passed ✅")

## ✅ Milestone-3 checks

- [x] Leak-free pipeline (preprocessing inside, fitted on train only)
- [x] Stratified split with fixed seed
- [x] From-scratch metrics reported (accuracy/precision/recall/F1)
- [x] Model artefact saved to `outputs/`

**Next:** `04_pytorch_cnn.ipynb` — the same classify-evaluate-save loop with neural nets.